# 🌍 Ekegusii Multilingual NMT: Google Colab Cloudflare Free Tunnel Deployment
This notebook provides a 1-click solution to deploy the fine-tuned **Ekegusii LLM Translation Model** on **Google Colab** using **Cloudflare Free Tunnel (`cloudflared`)**, **Hugging Face Hub** (`aykgeh/Ekegusii-LLM-Translation`), and **GitHub** for the HTML/CSS/JS web interface.

### 🚀 Key Features:
- **🤗 Model Source**: Loaded directly from Hugging Face Repo `aykgeh/Ekegusii-LLM-Translation` (Supports checkpoint subfolder `qwen/E1_English_Ekegusii/checkpoint-8000` & E10 Winner).
- **⚡ Cloudflare Free Tunnel**: Exposes your Colab GPU web server via a public, secure `https://*.trycloudflare.com` URL (No sign-up, no password, 100% free).
- **🎨 GitHub Web Assets**: Uses clean HTML5/CSS3 glassmorphism web interface (`index.html`, `style.css`, `app.js`) fetched from GitHub.
- **⚡ 4-Bit NF4 Quantization**: Runs on standard free Google Colab T4 GPU with ~4.5 GB VRAM usage.

## 🖥️ Step 1: Verify Google Colab GPU Environment
Ensure you have enabled GPU accelerator in Colab (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
import torch
import sys

print('=' * 60)
print('🔍 CHECKING COLAB GPU ACCELERATOR...')
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f'✅ GPU Detected   : {device_name} ({vram_gb} GB VRAM)')
else:
    print('⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.')

## 🔑 Step 2: Hugging Face Authentication
This step reads `HF_TOKEN` from Google Colab Secrets (the 🔑 icon on the left sidebar) or allows manual token entry.

In [ ]:
import os
from huggingface_hub import login

#@title 🔑 Hugging Face Token Setup { run: 'auto' }
HF_TOKEN_INPUT = "" #@param {type:"string"}

hf_token = None
# 1. Try reading from Colab Secrets
try:
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None
except Exception:
    hf_token = None

# 2. Fallback to manual entry
if not hf_token and HF_TOKEN_INPUT.strip():
    hf_token = HF_TOKEN_INPUT.strip()

if hf_token:
    login(token=hf_token.strip())
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("✅ Hugging Face Authentication Successful!")
else:
    print("ℹ️ Running in Public Model mode (No HF_TOKEN required for public repos).")

## 📦 Step 3: Install Required Packages & Download Cloudflare Tunnel (`cloudflared`)
We install `transformers`, `peft`, `bitsandbytes`, `fastapi`, `uvicorn`, and download the official **Cloudflare `cloudflared`** Linux binary.

In [ ]:
# 1. Install Python ML & Web dependencies
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pydantic streamlit nest_asyncio requests

# 2. Download and install Cloudflare Tunnel (cloudflared) debian package
print('⚡ Downloading Cloudflare Tunnel (cloudflared)...')
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb
!cloudflared --version
print('✅ Cloudflare Tunnel Binary Installed Successfully!')

## 📁 Step 4: Clone GitHub Repository (HTML, CSS, JS Frontend Assets)
We fetch the project repository from GitHub to access `web/static/index.html`, `web/static/style.css`, `web/static/app.js`, and `web/server.py`.

In [ ]:
import os

# Clone repository if running in Colab
REPO_URL = "https://github.com/aykahsay/Ekegusii-LLM-Translation.git"

if not os.path.exists('web'):
    print('📥 Cloning project repository from GitHub...')
    !git clone {REPO_URL} repo_temp
    if os.path.exists('repo_temp/web'):
        !cp -r repo_temp/web ./web
        !cp -r repo_temp/app.py ./app.py 2>/dev/null || true
        !rm -rf repo_temp
        print('✅ Web assets successfully extracted from GitHub!')
    else:
        print('⚠️ Using local web directory.')
else:
    print('✅ Directory `web/` already exists.')

## 🤗 Step 5: Load Base Model & Hugging Face Checkpoint Adapter
We load **`Qwen/Qwen2.5-7B-Instruct`** in 4-bit quantization and attach the fine-tuned LoRA checkpoint adapter from **`aykgeh/Ekegusii-LLM-Translation`**.

> 📍 **Specified Checkpoint**: `qwen/E1_English_Ekegusii/checkpoint-8000`
> 🏆 **Winner Model Option**: `qwen/E10_Model_B_English_Ekegusii`

In [ ]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Model identifiers
BASE_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
HF_REPO_ID = 'aykgeh/Ekegusii-LLM-Translation'
# Default checkpoint requested by user (E10 Winner Model)
CHECKPOINT_SUBFOLDER = 'qwen/E10_Model_B_English_Ekegusii/checkpoint-8000'

token_value = os.environ.get('HF_TOKEN', None)
device_map = {"": 0} if torch.cuda.is_available() else "auto"

print('⏳ 1/3 Loading 4-bit Quantization Config...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

print('⏳ 2/3 Loading Qwen2.5-7B Base Model & Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, padding_side='left', token=token_value)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.bfloat16,
    token=token_value
)

print(f"⏳ 3/3 Attaching Adapter Checkpoint ('{CHECKPOINT_SUBFOLDER}') from Hugging Face Hub...")
peft_model = PeftModel.from_pretrained(
    base_model,
    HF_REPO_ID,
    subfolder=CHECKPOINT_SUBFOLDER,
    token=token_value
)
peft_model.eval()

# Register model in server cache (Bulletproof web module resolution)
import sys, os

if not any(os.path.exists(os.path.join(p, 'web')) for p in ['.', 'repo', '/content/repo', os.getcwd()]):
    print('📥 Cloning project repository from GitHub...')
    !git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git repo

for candidate in [os.getcwd(), os.path.abspath('repo'), '/content/repo', '/content']:
    if os.path.exists(os.path.join(candidate, 'web')):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)

from web.server import MODEL_CACHE
MODEL_CACHE['base_model'] = base_model
MODEL_CACHE['tokenizer'] = tokenizer
MODEL_CACHE['active_peft'] = peft_model
MODEL_CACHE['active_subfolder'] = CHECKPOINT_SUBFOLDER

print('✅ Hugging Face Checkpoint Successfully Loaded on GPU and Registered in Web Cache!')

## 🚀 Step 6: Launch FastAPI Web Backend
Starts the Uvicorn web server in a background process listening on `http://127.0.0.1:8000`.

In [ ]:
import subprocess, time

print('🌐 Starting FastAPI Web Server on port 8000...')
server_process = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'web.server:app', '--host', '127.0.0.1', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print('✅ FastAPI Backend Server active on port 8000!')

## ⚡ Step 7: Launch Cloudflare Free Tunnel (`cloudflared`) & Get Public HTTPS Link
Cloudflare Tunnel establishes an encrypted bridge to your Colab server and provides a free temporary public URL (`https://*.trycloudflare.com`).

In [ ]:
import subprocess, re, time
from IPython.display import HTML, display

# Start cloudflared tunnel in background
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('⚡ Initializing Cloudflare Tunnel...')
public_url = None

# Wait and parse cloudflared output for the trycloudflare URL
for i in range(30):
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #0f172a, #1e293b); padding: 24px; border-radius: 16px; border: 2px solid #3b82f6; text-align: center; box-shadow: 0 10px 30px rgba(0,0,0,0.5); font-family: sans-serif;">
        <h2 style="color: #60a5fa; margin-bottom: 8px;">⚡ Cloudflare Live Tunnel Active!</h2>
        <p style="color: #cbd5e1; font-size: 1.1em;">Click the link below to open your deployed Ekegusii NMT Web Application:</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #3b82f6, #8b5cf6); color: white; padding: 14px 28px; border-radius: 10px; font-weight: bold; font-size: 1.2em; text-decoration: none; margin: 16px 0; box-shadow: 0 4px 15px rgba(59,130,246,0.4);">
            🌐 Open Web Portal: {public_url}
        </a>
        <p style="color: #94a3b8; font-size: 0.85em; margin-top: 10px;">
            🤗 Hugging Face Model Repo: <strong>aykgeh/Ekegusii-LLM-Translation</strong><br>
            📍 Loaded Checkpoint: <strong>{CHECKPOINT_SUBFOLDER}</strong>
        </p>
    </div>
    """))
else:
    print('⚠️ Tunnel setup delayed. Checking tunnel process log...')
    print(tunnel_process.stdout.read())

## 🧪 Step 8: Live Interactive Translation Sandbox (Inside Notebook)
You can test sentence translation directly inside Colab notebook cells while your Cloudflare web application is running.

In [ ]:
def translate_sentence(text: str, source_lang: str = 'English', target_lang: str = 'Ekegusii') -> str:
    """Translate text directly using loaded PyTorch model."""
    prompt = f'<|im_start|>user\nTranslate {source_lang} to {target_lang}:\n{text}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(peft_model.device)
    
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Test Public Service Announcement (PSA) ---
test_sentence = "Please wash your hands regularly with clean running water and soap to prevent cholera infection."
result = translate_sentence(test_sentence, source_lang="English", target_lang="Ekegusii")

print('=' * 70)
print(f'INPUT (EN):  {test_sentence}')
print(f'OUTPUT (EKE): {result}')
print('=' * 70)